# Machine Learning and Natural Language Processing
## Esercizio 1. Embedding contestuali: congelare o addestrare?

**Master AIDA, Edizione XVI, 2026.** Giuliano Lorenzo, PhD Student in Economics, Statistics and Data Science

[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/giulianolorenzo-PhD/aida-ml-nlp/blob/main/esercizio_1.ipynb)

Nella lezione scorsa abbiamo usato un modello contestuale già addestrato sul sentiment dei tweet, senza modificarlo.
Così com'è sbagliava quasi tutte le recensioni neutre (**il suo "neutro" non era il nostro**);
usato come estrattore di feature batteva TF-IDF, ma di poco.

Oggi rispondiamo a due domande:

1. **Da dove partire?** Da un modello già addestrato su un compito simile al nostro o da un modello generale?
2. **Come adattarlo?** Lo teniamo **congelato** (frozen) e usiamo i suoi vettori come feature (feature extraction),
   oppure lo **riaddestriamo** sui nostri dati (fine-tuning)?

|  | Congelato | Fine-tuning |
|:--|:--|:--|
| **Specializzato** (RoBERTa addestrato sul sentiment dei tweet) | sezione 2 | sezione 3 |
| **Generale** (RoBERTa base) | sezione 2 | sezione 3 |

Nella sezione 2 aggiungiamo un terzo estrattore congelato: un modello moderno, nato per produrre embedding di frasi.

Vedremo anche come **salvare e ricaricare** modelli e stati di addestramento, **quanti dati** servono e **quanto costa** ogni scelta.

## Indice
0. Setup
1. Il punto di partenza: TF-IDF
2. Modelli congelati: i vettori come feature
3. Fine-tuning: il modello impara il nostro compito
4. Salvare e ricaricare
5. Il confronto
6. Quanti dati servono?
7. Quanto costa?
8. Il test, una sola volta
9. Cosa portare a casa

## 0. Setup

- **Attivate la GPU**: menu *Runtime*, *Cambia tipo di runtime*, *GPU T4*. Senza GPU il fine-tuning richiede ore.
- I tempi indicati nel testo sono stime per la GPU T4: variano con la macchina assegnata da Colab.
- Ogni risultato costoso viene **salvato su file**: se la sessione si riavvia, il notebook lo ricarica invece di ricalcolarlo.

### 0.1 Librerie e impostazioni

In [1]:
import json
import math
import re
import shutil
import time
import warnings
from pathlib import Path

import joblib
import kagglehub
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
import torch
from huggingface_hub.utils import logging as hub_logging
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from transformers import (AutoConfig, AutoModel, AutoModelForSequenceClassification, AutoTokenizer,
                          DataCollatorWithPadding, Trainer, TrainingArguments)
from transformers.utils import logging as transformers_logging

transformers_logging.set_verbosity_error()      # nasconde i messaggi tecnici di transformers
transformers_logging.disable_progress_bar()      # e le barre di avanzamento durante il caricamento dei modelli
hub_logging.set_verbosity_error()                # nasconde gli avvisi tecnici dell'Hugging Face Hub
warnings.filterwarnings("ignore", message="Unknown solver options")   # avviso innocuo tra versioni di scipy e scikit-learn
pio.templates.default = "plotly_white"          # stile grafico pulito
RANDOM_STATE = 42                               # seme per la riproducibilità
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Dispositivo: {DEVICE}", f"({torch.cuda.get_device_name(0)})" if DEVICE == "cuda" else "")

Dispositivo: cuda (NVIDIA GeForce RTX 4080 Laptop GPU)


In Colab i file della sessione **spariscono** quando il runtime si chiude. Con `USE_DRIVE = True`
vengono salvati su Google Drive e restano disponibili (utile per il project work).

In [2]:
USE_DRIVE = False

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    SAVE_DIR = Path("/content/drive/MyDrive/aida_esercizio_1")
else:
    SAVE_DIR = Path("salvataggi")

SAVE_DIR.mkdir(parents=True, exist_ok=True)
print(f"I salvataggi vanno in: {SAVE_DIR}")

I salvataggi vanno in: salvataggi


### 0.2 I dati

Stesso dataset e **stessa pulizia** della lezione scorsa, in una sola cella: eliminate le recensioni senza testo
e quelle duplicate; pulizia di base (`text_clean`, per i modelli contestuali) e normalizzazione (`text_norm`, per TF-IDF);
sentiment a 3 classi da `Rating` (1-2 negativo, 3 neutro, 4-5 positivo).

In [3]:
df = pd.read_csv(next(Path(kagglehub.dataset_download("nicapotato/womens-ecommerce-clothing-reviews")).glob("*.csv")),
                 index_col=0)
df = df.dropna(subset=["Review Text"]).drop_duplicates(subset="Review Text")

ENCODING_FIXES = {"ã©": "é", "ã¨": "è", "ã¼": "ü", "â": ""}
CONTRACTIONS = {r"\bwon't\b": "will not", r"\bcan't\b": "can not", r"n't\b": " not", r"'re\b": " are",
                r"'m\b": " am", r"'ll\b": " will", r"'ve\b": " have", r"'d\b": " would", r"'s\b": ""}


def clean_text(text):
    """Livello 1: corregge la codifica, gli spazi e l'ultima parola spezzata dei testi troncati."""
    for wrong, right in ENCODING_FIXES.items():
        text = text.replace(wrong, right)
    text = re.sub(r"\s+", " ", text).strip()
    if len(text) >= 490 and text[-1] not in ".!?)\"'":
        text = text.rsplit(" ", 1)[0]
    return text


def normalize_text(text):
    """Livello 2 (solo per TF-IDF): minuscole, niente numeri e punteggiatura, contrazioni espanse."""
    text = re.sub(r"\S*\d\S*", " ", text.lower())
    for pattern, replacement in CONTRACTIONS.items():
        text = re.sub(pattern, replacement, text)
    return re.sub(r"\s+", " ", re.sub(r"[^a-z\s]", " ", text)).strip()


df["text_clean"] = df["Review Text"].map(clean_text)
df["text_norm"] = df["text_clean"].map(normalize_text)

CLASSES = ["negativo", "neutro", "positivo"]                          # l'ordine definisce le etichette 0, 1, 2
df["label"] = df["Rating"].map({1: 0, 2: 0, 3: 1, 4: 2, 5: 2})
df["sentiment"] = pd.Categorical(df["label"].map(dict(enumerate(CLASSES))), categories=CLASSES, ordered=True)

print(f"Recensioni: {len(df)}")
df["sentiment"].value_counts(normalize=True, sort=False).mul(100).round(1).rename("%").to_frame()

Recensioni: 22634


,%
sentiment,
negativo,10.5
neutro,12.5
positivo,77.1


#### Addestramento, validazione e test

- **Test** (20%): lo stesso della lezione scorsa; resta chiuso fino alla sezione 8.
- **Validazione** (20% del resto): la lezione scorsa usava la cross-validation a 5 fold, ma con il fine-tuning
  significherebbe addestrare ogni modello 5 volte. Usiamo un'unica validazione, uguale per tutti i modelli.
- **Addestramento**: quello che rimane.

In [4]:
train_df, test_df = train_test_split(df, test_size=0.2, stratify=df["sentiment"], random_state=RANDOM_STATE)
fit_df, val_df = train_test_split(train_df, test_size=0.2, stratify=train_df["sentiment"], random_state=RANDOM_STATE)

print(f"Addestramento: {len(fit_df)}  Validazione: {len(val_df)}  Test: {len(test_df)}")

Addestramento: 14485  Validazione: 3622  Test: 4527


Misuriamo la **macro-F1** (la media delle F1 delle tre classi) e la F1 di ogni classe, in particolare del **neutro**.
Come nella lezione scorsa, differenze sotto 0,01 vanno lette come un pareggio.

In [5]:
results = {}


def record(name, y_true, y_pred, seconds=None):
    """Calcola macro-F1 e F1 per classe, le salva in `results` e le mostra in una riga."""
    f1_per_class = f1_score(y_true, y_pred, labels=[0, 1, 2], average=None)
    results[name] = {"Macro-F1": f1_per_class.mean(), **dict(zip(CLASSES, f1_per_class))}
    if seconds is not None:
        results[name]["Addestramento (s)"] = seconds
    return pd.DataFrame([results[name]], index=[name]).round(3)


def results_table():
    return pd.DataFrame.from_dict(results, orient="index")[["Macro-F1", *CLASSES]].sort_values("Macro-F1", ascending=False).round(3)

## 1. Il punto di partenza: TF-IDF

La rappresentazione statica migliore della lezione scorsa: TF-IDF su parole e coppie di parole, con la regressione logistica.
Scegliamo sulla validazione `C`, l'**inverso** della forza della regolarizzazione (`C` piccolo, modello più semplice).

Scegliere `C` sugli stessi dati su cui misuriamo dà un piccolo vantaggio a TF-IDF e, nella sezione 2, ai modelli congelati.
Il fine-tuning invece non sceglie nulla sulla validazione: se vince, la vittoria è solida.

In [6]:
start = time.time()
tfidf_models = {}
for C in [0.3, 1, 3]:
    tfidf_models[C] = make_pipeline(
        TfidfVectorizer(min_df=2, sublinear_tf=True, ngram_range=(1, 2)),
        LogisticRegression(C=C, max_iter=2000, class_weight="balanced"),
    ).fit(fit_df["text_norm"], fit_df["label"])
seconds = (time.time() - start) / len(tfidf_models)                  # tempo medio di un addestramento

val_f1 = {C: f1_score(val_df["label"], m.predict(val_df["text_norm"]), average="macro") for C, m in tfidf_models.items()}
best_C = max(val_f1, key=val_f1.get)
tfidf_model = tfidf_models[best_C]
print("Macro-F1 in validazione per C:", {C: round(f, 3) for C, f in val_f1.items()}, f"  scelto C = {best_C}")

record("TF-IDF", val_df["label"], tfidf_model.predict(val_df["text_norm"]), seconds)

Macro-F1 in validazione per C: {0.3: 0.641, 1: 0.649, 3: 0.641}   scelto C = 1


,Macro-F1,negativo,neutro,positivo,Addestramento (s)
TF-IDF,0.649,0.572,0.45,0.926,1.106


#### Salvare un modello di scikit-learn

Con `joblib` il modello addestrato, **vettorizzatore compreso** (la pipeline li tiene insieme), diventa un file
che si ricarica identico: niente riaddestramento, e si può usare altrove (un'applicazione, un server).
Attenzione: caricare un file `joblib` può eseguire codice. Caricate solo file di cui vi fidate.

In [7]:
path = SAVE_DIR / "tfidf_logreg.joblib"
joblib.dump(tfidf_model, path)
reloaded = joblib.load(path)

same = (reloaded.predict(val_df["text_norm"]) == tfidf_model.predict(val_df["text_norm"])).all()
print(f"File: {path.name}, {path.stat().st_size / 1e6:.1f} MB. Previsioni identiche dopo il caricamento: {same}")

File: tfidf_logreg.joblib, 3.0 MB. Previsioni identiche dopo il caricamento: True


## 2. Modelli congelati: i vettori come feature

Due modelli con la **stessa architettura** (RoBERTa base: 12 strati, 125 milioni di parametri) e lo **stesso tokenizer**:
cambia solo ciò che hanno imparato. Il confronto è **controllato**: le differenze dipendono dal punto di partenza.

- **Specializzato** (`cardiffnlp/twitter-roberta-base-sentiment-latest`): RoBERTa base preaddestrato anche su circa
  124 milioni di tweet, poi addestrato a classificarne il sentiment (negativo, neutro, positivo).
- **Generale** (`FacebookAI/roberta-base`): il modello di partenza, preaddestrato solo a indovinare parole nascoste
  (masked language modeling). Sa "leggere l'inglese", ma non conosce il sentiment.

#### Da una sequenza di vettori a un vettore (pooling)

Il modello restituisce **un vettore di 768 numeri per ogni token**; per classificare una recensione ne serve uno solo.
- **`<s>`**: il vettore del token speciale iniziale, quello che la testa di classificazione usa per decidere;
- **media**: la media dei vettori di tutti i token, esclusi quelli di riempimento (padding).

In [8]:
MODELS = {"Specializzato": "cardiffnlp/twitter-roberta-base-sentiment-latest",
          "Generale": "FacebookAI/roberta-base"}
MAX_LENGTH = 128                     # token per recensione: il 97% delle recensioni ci sta per intero


@torch.no_grad()                     # nessun addestramento: niente gradienti, meno memoria
def encode(model_name, texts, batch_size=64):
    """Vettori contestuali delle recensioni con due pooling: il token <s> e la media dei token."""
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(DEVICE).eval()   # solo il corpo, senza testa di classificazione
    first, mean = [], []
    for start in range(0, len(texts), batch_size):
        batch = tokenizer(texts[start:start + batch_size], padding=True, truncation=True,
                          max_length=MAX_LENGTH, return_tensors="pt").to(DEVICE)
        with torch.autocast(DEVICE, enabled=DEVICE == "cuda"):         # precisione ridotta sulla GPU: più veloce
            hidden = model(**batch).last_hidden_state.float()          # (recensioni, token, 768)
        mask = batch["attention_mask"].unsqueeze(-1)                   # 1 = token vero, 0 = padding
        first.append(hidden[:, 0].cpu())
        mean.append(((hidden * mask).sum(1) / mask.sum(1)).cpu())
    return {"<s>": torch.cat(first).numpy(), "media": torch.cat(mean).numpy()}

Calcoliamo i vettori di addestramento e validazione e li **salviamo** in un file `.npz` (il formato compresso di NumPy):
alla seconda esecuzione si ricaricano in un istante. **Circa 3 minuti con la GPU T4** (stima).

In [9]:
vectors = {}
for key, model_name in MODELS.items():
    path = SAVE_DIR / f"vettori_{key.lower()}.npz"
    if not path.exists():
        start = time.time()
        enc = {split: encode(model_name, data["text_clean"].tolist()) for split, data in [("fit", fit_df), ("val", val_df)]}
        np.savez_compressed(path, **{f"{split}|{pool}": X for split, d in enc.items() for pool, X in d.items()})
        print(f"{key}: vettori calcolati in {time.time() - start:.0f} s")
    vectors[key] = dict(np.load(path))

print({k: v.shape for k, v in vectors["Generale"].items()})

Specializzato: vettori calcolati in 19 s


Generale: vettori calcolati in 19 s


{'fit|<s>': (14485, 768), 'fit|media': (14485, 768), 'val|<s>': (3622, 768), 'val|media': (3622, 768)}


Su ogni insieme di vettori addestriamo lo stesso classificatore: una regressione logistica sulle feature standardizzate,
con `C` scelto sulla validazione.

In [10]:
def fit_on_vectors(X_fit, y_fit, X_val, y_val, Cs=(0.003, 0.01, 0.03)):
    """Regressione logistica con il C migliore in validazione. Restituisce le previsioni sulla validazione."""
    best_score, best_pred = -1, None
    for C in Cs:
        model = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=2000, class_weight="balanced"))
        pred = model.fit(X_fit, y_fit).predict(X_val)
        score = f1_score(y_val, pred, average="macro")
        if score > best_score:
            best_score, best_pred = score, pred
    return best_pred


for key in MODELS:
    for pool in ["<s>", "media"]:
        pred = fit_on_vectors(vectors[key][f"fit|{pool}"], fit_df["label"], vectors[key][f"val|{pool}"], val_df["label"])
        record(f"{key}, congelato ({pool})", val_df["label"], pred)

results_table()

,Macro-F1,negativo,neutro,positivo
"Generale, congelato (media)",0.674,0.615,0.480,0.926
"Specializzato, congelato (media)",0.668,0.602,0.479,0.923
"Specializzato, congelato (<s>)",0.661,0.606,0.462,0.916
"Generale, congelato (<s>)",0.654,0.592,0.449,0.920
TF-IDF,0.649,0.572,0.450,0.926


- I vettori congelati di RoBERTa superano TF-IDF al massimo di circa 0,025: un guadagno **piccolo**.
- La **media** è sempre almeno pari a `<s>`, e per il generale è nettamente meglio.
- Da congelati, specializzato e generale sono **alla pari**: aver imparato il sentiment dei tweet
  non produce vettori migliori per le nostre recensioni.

#### Un terzo estrattore: un modello nato per gli embedding

RoBERTa è preaddestrato a indovinare parole, non a riassumere una frase in un vettore. I **modelli di embedding**
(sentence embedding) sono addestrati proprio per questo: testi simili, vettori vicini.
Per sceglierne uno abbiamo consultato **MTEB** (Massive Text Embedding Benchmark), la classifica di riferimento
di questi modelli, guardando i compiti di **classificazione**; poi abbiamo confrontato alcuni candidati sulla nostra validazione.
Il migliore è `Alibaba-NLP/gte-modernbert-base` (149 milioni di parametri, 2025): lo usiamo congelato, come i RoBERTa.
**Circa 1 minuto con la GPU T4** (stima).

In [11]:
EMBEDDING_MODEL = "Alibaba-NLP/gte-modernbert-base"
path = SAVE_DIR / "vettori_embedding.npz"
if not path.exists():
    embedder = SentenceTransformer(EMBEDDING_MODEL, device=DEVICE,
                                   model_kwargs={"torch_dtype": torch.float16} if DEVICE == "cuda" else {})
    start = time.time()
    np.savez_compressed(path, **{split: embedder.encode(data["text_clean"].tolist(), batch_size=64, normalize_embeddings=True)
                                 for split, data in [("fit", fit_df), ("val", val_df)]})
    print(f"Vettori calcolati in {time.time() - start:.0f} s")
embedding_vectors = dict(np.load(path))

pred = fit_on_vectors(embedding_vectors["fit"], fit_df["label"], embedding_vectors["val"], val_df["label"])
record("Embedding moderno, congelato", val_df["label"], pred)
results_table()

Vettori calcolati in 19 s


,Macro-F1,negativo,neutro,positivo
"Embedding moderno, congelato",0.698,0.654,0.504,0.935
"Generale, congelato (media)",0.674,0.615,0.480,0.926
"Specializzato, congelato (media)",0.668,0.602,0.479,0.923
"Specializzato, congelato (<s>)",0.661,0.606,0.462,0.916
"Generale, congelato (<s>)",0.654,0.592,0.449,0.920
TF-IDF,0.649,0.572,0.450,0.926


Il modello di embedding, **congelato** e con lo stesso classificatore, supera entrambi i RoBERTa congelati:
un modello nato per produrre vettori di frase dà feature migliori. Nella sezione 5 lo confrontiamo con il fine-tuning.

## 3. Fine-tuning: il modello impara il nostro compito

Nel fine-tuning si aggiornano **tutti i 125 milioni di pesi**, non solo il classificatore finale.
Sopra il modello c'è una **testa di classificazione** (classification head): una piccola rete a due strati
che trasforma il vettore di `<s>` in tre punteggi, uno per classe.

**Perché `<s>`, se da congelati la media era meglio?** Nel fine-tuning cambiano tutti i pesi: il modello impara
a raccogliere in `<s>`, che con l'attenzione vede tutti i token, ciò che serve al compito. Il generale congelato
non l'aveva mai imparato: il suo preaddestramento non chiede a `<s>` di riassumere la frase.

- Modello **specializzato**: la testa esiste già, con le nostre tre classi nello stesso ordine (lo verifichiamo qui sotto).
- Modello **generale**: la testa non esiste; viene creata con **pesi casuali** e imparata da zero.

| Iperparametro | Valore | Perché |
|:--|:--|:--|
| Learning rate | 0,00002 (2e-5) | passi piccoli: si ritocca ciò che il modello sa, senza cancellarlo |
| Epoche | 2 | ogni recensione viene vista 2 volte; con troppe epoche il modello impara a memoria (overfitting) |
| Batch | 32 | i pesi si aggiornano dopo ogni gruppo di 32 recensioni |
| Warmup | 10% dei passi | il learning rate sale da 0 al valore scelto, poi scende linearmente fino a 0 |
| Precisione mista (fp16) | sì | molti calcoli a 16 bit invece di 32: più veloce e meno memoria sulla GPU |

In [12]:
print("Classi della testa del modello specializzato:", AutoConfig.from_pretrained(MODELS["Specializzato"]).id2label)

Classi della testa del modello specializzato: {0: 'negative', 1: 'neutral', 2: 'positive'}


In [13]:
def to_dataset(tokenizer, data):
    """Una lista di esempi tokenizzati: è tutto ciò che serve al Trainer."""
    enc = tokenizer(data["text_clean"].tolist(), truncation=True, max_length=MAX_LENGTH)
    return [{"input_ids": ids, "attention_mask": mask, "labels": label}
            for ids, mask, label in zip(enc["input_ids"], enc["attention_mask"], data["label"])]


def macro_f1(eval_pred):
    """Metrica calcolata dal Trainer sulla validazione."""
    return {"macro_f1": f1_score(eval_pred.label_ids, eval_pred.predictions.argmax(-1), average="macro")}


def finetune(model_name, train_data, epochs=2, output_dir="checkpoint", per_epoch=True, batch_size=32, lr=2e-5):
    """Prepara il fine-tuning per le nostre tre classi. Restituisce il Trainer e il tokenizer.
    per_epoch=True: a ogni epoca valuta sulla validazione e salva un checkpoint."""
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=3, id2label=dict(enumerate(CLASSES)), label2id={c: i for i, c in enumerate(CLASSES)})
    steps = epochs * math.ceil(len(train_data) / batch_size)              # aggiornamenti totali dei pesi
    args = TrainingArguments(
        output_dir=output_dir, num_train_epochs=epochs, learning_rate=lr, per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=128, warmup_steps=int(0.1 * steps), weight_decay=0.01,
        eval_strategy="epoch" if per_epoch else "no", save_strategy="epoch" if per_epoch else "no",
        save_total_limit=1,                                              # tiene solo l'ultimo checkpoint
        fp16=DEVICE == "cuda", seed=RANDOM_STATE, logging_steps=50, report_to="none",
        disable_tqdm=False)                                              # tabella di avanzamento durante l'addestramento
    trainer = Trainer(model=model, args=args, train_dataset=to_dataset(tokenizer, train_data),
                      eval_dataset=to_dataset(tokenizer, val_df), data_collator=DataCollatorWithPadding(tokenizer),
                      compute_metrics=macro_f1)
    return trainer, tokenizer


@torch.no_grad()
def predict_proba(model, tokenizer, texts, batch_size=128, device=DEVICE):
    """Probabilità delle tre classi per una lista di testi."""
    model = model.to(device).eval()
    probs = []
    for start in range(0, len(texts), batch_size):
        batch = tokenizer(texts[start:start + batch_size], padding=True, truncation=True,
                          max_length=MAX_LENGTH, return_tensors="pt").to(device)
        probs.append(model(**batch).logits.float().softmax(-1).cpu())
    return torch.cat(probs).numpy()

Prima di addestrare misuriamo la macro-F1 **senza addestramento**: per lo specializzato è il modello pronto all'uso
della lezione scorsa, per il generale una testa casuale. Durante l'addestramento una tabella mostra, a ogni epoca,
la loss e la macro-F1 sulla validazione. Se un modello è già stato salvato (sezione 4), lo ricarichiamo.
**Circa 5 minuti per modello con la GPU T4** (stima).

In [14]:
finetuned, histories = {}, {}
for key, model_name in MODELS.items():
    model_dir = SAVE_DIR / f"modello_{key.lower()}"
    if (model_dir / "risultati.json").exists():                          # già addestrato: lo ricarichiamo
        info = json.loads((model_dir / "risultati.json").read_text())
        model = AutoModelForSequenceClassification.from_pretrained(model_dir)
        tokenizer = AutoTokenizer.from_pretrained(model_dir)
    else:
        trainer, tokenizer = finetune(model_name, fit_df, output_dir=f"checkpoint_{key.lower()}")
        before = trainer.evaluate()["eval_macro_f1"]                     # prima del fine-tuning
        start = time.time()
        trainer.train()
        info = {"prima": before, "tempo_s": time.time() - start,
                "storia": [h for h in trainer.state.log_history if "eval_macro_f1" in h or "loss" in h]}
        model = trainer.model
        trainer.save_model(model_dir)                                    # pesi e configurazione (sezione 4)
        tokenizer.save_pretrained(model_dir)
        (model_dir / "risultati.json").write_text(json.dumps(info))
    finetuned[key], histories[key] = (model, tokenizer), info
    pred = predict_proba(model, tokenizer, val_df["text_clean"].tolist()).argmax(1)
    print(f"{key}: macro-F1 prima del fine-tuning {info['prima']:.3f}, tempo di addestramento {info['tempo_s'] / 60:.1f} min")
    display(record(f"{key}, fine-tuning", val_df["label"], pred, info["tempo_s"]))

Training Loss,Validation Loss,Epoch,Macro F1
No log,0.497657,0,0.547025


Epoch,Training Loss,Validation Loss,Macro F1
1,0.318891,0.316920,0.704727
2,0.267229,0.319008,0.722866


Specializzato: macro-F1 prima del fine-tuning 0.547, tempo di addestramento 1.4 min


,Macro-F1,negativo,neutro,positivo,Addestramento (s)
"Specializzato, fine-tuning",0.723,0.666,0.55,0.953,86.669


Training Loss,Validation Loss,Epoch,Macro F1
No log,1.060873,0,0.287811


Epoch,Training Loss,Validation Loss,Macro F1
1,0.330804,0.341557,0.666313
2,0.278065,0.328298,0.695915


Generale: macro-F1 prima del fine-tuning 0.288, tempo di addestramento 1.4 min


,Macro-F1,negativo,neutro,positivo,Addestramento (s)
"Generale, fine-tuning",0.696,0.626,0.513,0.949,86.483


La **loss** (l'errore che l'addestramento minimizza) sul training, ogni 50 passi, e la macro-F1 sulla validazione a fine epoca.

In [15]:
curves = pd.concat([pd.DataFrame(h["storia"]).assign(Modello=k) for k, h in histories.items()])
fig = px.line(curves.dropna(subset=["loss"]), x="epoch", y="loss", color="Modello",
              title="Loss di training durante il fine-tuning", labels={"epoch": "Epoca", "loss": "Loss"})
fig.show()
curves.dropna(subset=["eval_macro_f1"]).pivot(index="epoch", columns="Modello", values="eval_macro_f1").round(3)

Modello,Generale,Specializzato
epoch,,
1.0,0.666,0.705
2.0,0.696,0.723


- **Prima** del fine-tuning: lo specializzato vale circa 0,55 (è il modello pronto all'uso); il generale circa 0,29,
  lo stesso valore del modello pigro della lezione scorsa: con una testa casuale non sa ancora nulla del compito.
- **Dopo** due epoche: lo specializzato supera 0,71, il generale arriva intorno a 0,69-0,70.
- Il generale migliora ancora molto dalla prima alla seconda epoca: con più epoche potrebbe avvicinarsi.
  Va verificato sulla validazione, per accorgersi dell'overfitting.

## 4. Salvare e ricaricare

Addestrare costa tempo e denaro: un modello addestrato va **salvato**. Serve a
- **non rifare** il lavoro (la sessione di Colab si chiude, il PC si spegne);
- **usarlo altrove**: in un'applicazione, su un server, nel project work;
- **riprodurre** i risultati, anche a distanza di mesi;
- **condividerlo** con i colleghi (per esempio sull'Hugging Face Hub, con `push_to_hub`).

| Cosa | Contenuto | Serve per |
|:--|:--|:--|
| **Modello** (`save_model`, `save_pretrained`) | pesi, configurazione, tokenizer | usare il modello (inferenza) |
| **Checkpoint** (salvato dal Trainer a ogni epoca) | il modello più lo stato dell'addestramento | **riprendere** un addestramento interrotto |

In [16]:
def folder_size(folder):
    """Dimensione dei file di una cartella, in MB."""
    return pd.Series({f.name: round(f.stat().st_size / 1e6, 1) for f in sorted(Path(folder).iterdir()) if f.is_file()},
                     name="MB")


model_files = folder_size(SAVE_DIR / "modello_specializzato")
model_files.to_frame()

,MB
config.json,0.0
model.safetensors,498.6
risultati.json,0.0
tokenizer.json,3.6
tokenizer_config.json,0.0
training_args.bin,0.0


- `model.safetensors`: i **pesi**, 125 milioni di numeri da 4 byte (circa 500 MB). Il formato `safetensors` contiene
  solo numeri: caricarlo non esegue codice;
- `config.json`: l'**architettura** e i nomi delle classi;
- `tokenizer.json`, `tokenizer_config.json`: il **tokenizer**, da salvare sempre insieme al modello;
- `training_args.bin`: gli iperparametri dell'addestramento; `risultati.json`: le nostre metriche.

Ricaricare è la stessa riga usata per scaricare un modello dall'Hub, con la cartella al posto del nome.
Controlliamo che pesi e previsioni siano identici.

In [17]:
model_dir = SAVE_DIR / "modello_specializzato"
reloaded_model = AutoModelForSequenceClassification.from_pretrained(model_dir)
reloaded_tokenizer = AutoTokenizer.from_pretrained(model_dir)

original_weights = finetuned["Specializzato"][0].state_dict()
same_weights = all(torch.equal(original_weights[name].cpu(), w) for name, w in reloaded_model.state_dict().items())
sample_texts = val_df["text_clean"].tolist()[:256]
same_predictions = (predict_proba(*finetuned["Specializzato"], sample_texts).argmax(1)
                    == predict_proba(reloaded_model, reloaded_tokenizer, sample_texts).argmax(1)).all()
print(f"Pesi identici: {same_weights}. Previsioni identiche: {same_predictions}")

Pesi identici: True. Previsioni identiche: True


Il **checkpoint** serve a riprendere un addestramento interrotto esattamente da dove si era fermato. Oltre al modello contiene:
- `optimizer.pt`: lo stato dell'ottimizzatore AdamW, due numeri per ogni peso (per questo pesa il doppio del modello);
- `scheduler.pt`, `scaler.pt`, `rng_state.pth`: lo stato del learning rate, della precisione mista e del generatore casuale;
- `trainer_state.json`: a che punto era l'addestramento e la storia delle metriche.

In [18]:
checkpoints = sorted(Path("checkpoint_specializzato").glob("checkpoint-*"))
checkpoint_files = folder_size(checkpoints[-1]) if checkpoints else pd.Series(dtype=float)
if checkpoints:
    print(f"Checkpoint: {checkpoints[-1].name}")
    display(checkpoint_files.to_frame())
else:
    print("Nessun checkpoint in questa sessione: il modello è stato ricaricato dalla cartella salvata.")

Checkpoint: checkpoint-906


,MB
config.json,0.0
model.safetensors,498.6
optimizer.pt,997.4
rng_state.pth,0.0
scaler.pt,0.0
scheduler.pt,0.0
tokenizer.json,3.6
tokenizer_config.json,0.0
trainer_state.json,0.0
training_args.bin,0.0


Per riprendere un addestramento interrotto si ricrea il `Trainer` con gli stessi argomenti e si scrive:

```python
trainer.train(resume_from_checkpoint=True)     # riparte dall'ultimo checkpoint nella output_dir
```

I checkpoint sono grandi: finito l'addestramento si tiene il modello e si cancellano.

In [19]:
for folder in Path(".").glob("checkpoint_*"):
    shutil.rmtree(folder)
print("Checkpoint cancellati: restano i modelli salvati in", SAVE_DIR)

Checkpoint cancellati: restano i modelli salvati in salvataggi


## 5. Il confronto

### 5.1 La griglia due per due

In [20]:
best_frozen = {k: max(results[f"{k}, congelato ({p})"]["Macro-F1"] for p in ["<s>", "media"]) for k in MODELS}
grid = pd.DataFrame({"Congelato (miglior pooling)": best_frozen,
                     "Fine-tuning": {k: results[f"{k}, fine-tuning"]["Macro-F1"] for k in MODELS}})

fig = px.imshow(grid, text_auto=".3f", color_continuous_scale="Blues", zmin=0.6, zmax=0.75, aspect="auto",
                title=f"Macro-F1 in validazione (TF-IDF: {results['TF-IDF']['Macro-F1']:.3f})")
fig.update_layout(coloraxis_showscale=False)
fig.show()
print("Guadagno del fine-tuning:", {k: round(float(grid.loc[k, "Fine-tuning"] - best_frozen[k]), 3) for k in MODELS})

Guadagno del fine-tuning: {'Specializzato': 0.055, 'Generale': 0.022}


Leggiamo la griglia in due direzioni:
- **colonne** (come adattare): il fine-tuning migliora entrambi i modelli, lo specializzato di più;
- **righe** (da dove partire): da congelati è un pareggio, dopo il fine-tuning lo specializzato è avanti.

Le due scelte **interagiscono**: il punto di partenza conta quando il modello viene adattato.
Il migliore è il modello **specializzato, adattato** ai nostri dati.

Il modello di embedding congelato (sezione 2) arriva al livello del fine-tuning del generale, senza addestrare il modello.

### 5.2 Classe per classe

Quattro modelli: TF-IDF, il miglior congelato e i due fine-tuning.

In [21]:
best_frozen_name = max((n for n in results if "congelato" in n), key=lambda n: results[n]["Macro-F1"])
shown = ["TF-IDF", best_frozen_name, "Specializzato, fine-tuning", "Generale, fine-tuning"]
per_class = (results_table().loc[shown, CLASSES].reset_index(names="Modello")
             .melt(id_vars="Modello", var_name="Classe", value_name="F1"))
fig = px.bar(per_class, x="Classe", y="F1", color="Modello", barmode="group", text="F1",
             title="F1 per classe in validazione")
fig.update_traces(texttemplate="%{text:.2f}")
fig.show()

Il fine-tuning dello specializzato migliora soprattutto le due classi difficili: la F1 del **neutro** passa da circa 0,45
(TF-IDF) a circa 0,55 e cresce anche il **negativo**. Il positivo era già facile per tutti.

### 5.3 Le frasi difficili

Tre frasi della lezione scorsa e una con un pregio e un difetto: il modello pronto all'uso contro i due modelli adattati.

In [22]:
examples = ["this dress is not good",
            "not too tight, it fits perfectly",
            "i wanted to love this dress",
            "nice color, but the fabric feels cheap"]

ready = (AutoModelForSequenceClassification.from_pretrained(MODELS["Specializzato"]),
         AutoTokenizer.from_pretrained(MODELS["Specializzato"]))
rows = {"Pronto all'uso": ready, "Specializzato, fine-tuning": finetuned["Specializzato"],
        "Generale, fine-tuning": finetuned["Generale"]}
examples_table = pd.concat({name: pd.DataFrame(predict_proba(m, t, examples), index=examples, columns=CLASSES)
                            for name, (m, t) in rows.items()}, axis=1).round(2)
examples_table

Pronto all'uso                  \
                                             negativo neutro positivo   
this dress is not good                           0.93   0.06     0.01   
not too tight, it fits perfectly                 0.01   0.11     0.88   
i wanted to love this dress                      0.04   0.12     0.84   
nice color, but the fabric feels cheap           0.52   0.34     0.14   

                                       Specializzato, fine-tuning         \
                                                         negativo neutro   
this dress is not good                                       0.93   0.07   
not too tight, it fits perfectly                             0.00   0.00   
i wanted to love this dress                                  0.36   0.52   
nice color, but the fabric feels cheap                       0.22   0.71   

                                                Generale, fine-tuning         \
                                       positivo              negativo neutro   
this dress is not good                     0.00                  0.86   0.13   
not too tight, it fits perfectly           1.00                  0.00   0.00   
i wanted to love this dress                0.12                  0.16   0.31   
nice color, but the fabric feels cheap     0.07                  0.15   0.65   

                                                 
                                       positivo  
this dress is not good                     0.01  
not too tight, it fits perfectly           1.00  
i wanted to love this dress                0.53  
nice color, but the fabric feels cheap     0.20

- "i wanted to love this dress": positiva per il modello pronto all'uso, **neutra** per lo specializzato adattato.
- "nice color, but the fabric feels cheap": un pregio e un difetto, proprio il nostro neutro.
  Negativa per il modello pronto all'uso, **neutra** per entrambi i modelli adattati.

Quattro frasi aiutano a **capire** cosa è cambiato; per **misurare** c'è la validazione.

## 6. Quanti dati servono?

In azienda le recensioni etichettate sono spesso poche. Ripetiamo l'addestramento con **250, 1.000 e 4.000 recensioni**
(con le stesse proporzioni delle classi) e confrontiamo con tutte le 14.485.
Con pochi dati un'epoca dura pochi passi: per il fine-tuning usiamo abbastanza epoche da fare almeno 150 aggiornamenti dei pesi.
**Circa 6 minuti con la GPU T4** (stima); i risultati vengono salvati in un file.

In [23]:
SIZES = [250, 1000, 4000]
CURVE_MODELS = ["TF-IDF", "Specializzato, congelato (media)", "Embedding moderno, congelato",
                "Specializzato, fine-tuning", "Generale, fine-tuning"]


def train_and_predict(name, subset):
    """Addestra il modello `name` sulle recensioni di `subset` e restituisce le previsioni sulla validazione."""
    if name == "TF-IDF":
        model = make_pipeline(TfidfVectorizer(min_df=2, sublinear_tf=True, ngram_range=(1, 2)),
                              LogisticRegression(C=best_C, max_iter=2000, class_weight="balanced"))
        return model.fit(subset["text_norm"], subset["label"]).predict(val_df["text_norm"])
    rows = fit_df.index.get_indexer(subset.index)                        # le righe di subset tra i vettori già calcolati
    if name == "Specializzato, congelato (media)":
        return fit_on_vectors(vectors["Specializzato"]["fit|media"][rows], subset["label"],
                              vectors["Specializzato"]["val|media"], val_df["label"])
    if name == "Embedding moderno, congelato":
        return fit_on_vectors(embedding_vectors["fit"][rows], subset["label"], embedding_vectors["val"], val_df["label"])
    key = name.split(",")[0]                                             # "Specializzato" o "Generale"
    epochs = max(2, math.ceil(150 / math.ceil(len(subset) / 32)))         # almeno 150 aggiornamenti dei pesi
    trainer, _ = finetune(MODELS[key], subset, epochs=epochs, output_dir="checkpoint_curva", per_epoch=False)
    trainer.train()
    return trainer.predict(trainer.eval_dataset).predictions.argmax(-1)


curve_path = SAVE_DIR / "curva_apprendimento.json"
curve = json.loads(curve_path.read_text()) if curve_path.exists() else []
done = {(r["Modello"], r["Recensioni"]) for r in curve}
for n in SIZES:
    subset, _ = train_test_split(fit_df, train_size=n, stratify=fit_df["label"], random_state=RANDOM_STATE)
    for name in CURVE_MODELS:
        if (name, n) not in done:
            curve.append({"Modello": name, "Recensioni": n,
                          "Macro-F1": f1_score(val_df["label"], train_and_predict(name, subset), average="macro")})
            curve_path.write_text(json.dumps(curve))                     # salva dopo ogni punto
shutil.rmtree("checkpoint_curva", ignore_errors=True)

# L'ultimo punto della curva (tutte le recensioni) è già in `results`
curve_df = pd.concat([pd.DataFrame(curve), pd.DataFrame(
    [{"Modello": m, "Recensioni": len(fit_df), "Macro-F1": results[m]["Macro-F1"]} for m in CURVE_MODELS])])

fig = px.line(curve_df.sort_values("Recensioni"), x="Recensioni", y="Macro-F1", color="Modello", markers=True,
              log_x=True, title="Curva di apprendimento: macro-F1 in validazione al crescere dei dati")
fig.update_xaxes(tickvals=SIZES + [len(fit_df)])
fig.show()
curve_df.pivot(index="Modello", columns="Recensioni", values="Macro-F1").round(3)

Step,Training Loss
50,0.330409
100,0.054455
150,0.005519


Step,Training Loss
50,0.697825
100,0.285161
150,0.114405


Step,Training Loss
50,0.425831
100,0.310977
150,0.182849


Step,Training Loss
50,0.825983
100,0.443655
150,0.336756


Step,Training Loss
50,0.473309
100,0.400600
150,0.355953
200,0.301248
250,0.315149


Step,Training Loss
50,0.831671
100,0.478797
150,0.398543
200,0.344129
250,0.346305


Recensioni,250,1000,4000,14485
Modello,,,,
"Embedding moderno, congelato",0.650,0.663,0.691,0.698
"Generale, fine-tuning",0.566,0.625,0.668,0.696
"Specializzato, congelato (media)",0.627,0.627,0.653,0.668
"Specializzato, fine-tuning",0.652,0.661,0.665,0.723
TF-IDF,0.442,0.558,0.627,0.649


- Con **250 recensioni** TF-IDF crolla: troppo pochi esempi per imparare quali parole contano.
  I modelli contestuali sanno già leggere e restano molto più in alto.
- Il fine-tuning del **generale** soffre di più con pochi dati: parte da una testa casuale.
- Fino a 4.000 recensioni il **modello di embedding congelato** vale quanto il fine-tuning dello specializzato, o di più,
  a una frazione del costo. Con tutte le recensioni il fine-tuning dello specializzato torna davanti.

Ogni punto è un solo addestramento: con pochi dati differenze di 0,01-0,02 non sono affidabili.

## 7. Quanto costa?

In produzione contano anche **tempo**, **memoria** e **hardware**. Misuriamo il tempo per classificare 1.000 recensioni
sulla GPU e sulla CPU con 2 core (come una macchina di Colab senza GPU o un portatile modesto).

In [24]:
def seconds_per_1000(predict, texts):
    start = time.time()
    predict(texts)
    return (time.time() - start) / len(texts) * 1000


texts = val_df["text_clean"].tolist()[:500]
n_threads = torch.get_num_threads()
torch.set_num_threads(2)                                       # simula una CPU con 2 core
model, tokenizer = finetuned["Specializzato"]

costs = pd.DataFrame({
    "TF-IDF + regressione logistica": {
        "Parametri (milioni)": tfidf_model[-1].coef_.size / 1e6,
        "Spazio su disco (MB)": (SAVE_DIR / "tfidf_logreg.joblib").stat().st_size / 1e6,
        "Addestramento (s)": results["TF-IDF"]["Addestramento (s)"],
        "1.000 recensioni su CPU (s)": seconds_per_1000(lambda t: tfidf_model.predict([normalize_text(x) for x in t]), texts),
        "1.000 recensioni su GPU (s)": np.nan},
    "RoBERTa, fine-tuning": {
        "Parametri (milioni)": sum(p.numel() for p in model.parameters()) / 1e6,
        "Spazio su disco (MB)": model_files.sum(),
        "Addestramento (s)": results["Specializzato, fine-tuning"]["Addestramento (s)"],
        "1.000 recensioni su CPU (s)": seconds_per_1000(lambda t: predict_proba(model, tokenizer, t, 32, "cpu"), texts[:200]),
        "1.000 recensioni su GPU (s)": seconds_per_1000(lambda t: predict_proba(model, tokenizer, t), texts) if DEVICE == "cuda" else np.nan},
}).round(2)
torch.set_num_threads(n_threads)                               # ripristina i core disponibili
costs.astype(object).fillna("-")                               # "-": misura non applicabile

,TF-IDF + regressione logistica,"RoBERTa, fine-tuning"
Parametri (milioni),0.18,124.65
Spazio su disco (MB),2.96,502.2
Addestramento (s),1.11,86.67
1.000 recensioni su CPU (s),0.05,76.32
1.000 recensioni su GPU (s),-,1.19


- Il modello contestuale occupa **centinaia di volte** più spazio di TF-IDF.
- Su una CPU con 2 core è **più di mille volte** più lento: oltre un minuto per 1.000 recensioni contro una frazione di secondo.
- Con la GPU il tempo crolla, ma la GPU costa.

Prima di scegliere: quante previsioni al giorno? Dove gira il modello? Il guadagno in macro-F1 vale il costo?

## 8. Il test, una sola volta

Valutiamo sul test, mai usato finora, il fine-tuning migliore in validazione.
Riferimento della lezione scorsa: SVM lineare sui vettori congelati, macro-F1 0,672 sullo stesso test
(addestrata su tutto il training, 18.107 recensioni; il nostro modello ne ha viste 14.485).

In [25]:
best_key = max(MODELS, key=lambda k: results[f"{k}, fine-tuning"]["Macro-F1"])
best_name = f"{best_key}, fine-tuning"
print(f"Modello scelto: {best_name}. Migliore in assoluto in validazione: {results_table().index[0]}")
model, tokenizer = finetuned[best_key]

test_pred = predict_proba(model, tokenizer, test_df["text_clean"].tolist()).argmax(1)
test_f1 = f1_score(test_df["label"], test_pred, labels=[0, 1, 2], average=None)
test_result = {"Macro-F1": test_f1.mean(), **dict(zip(CLASSES, test_f1))}
pd.DataFrame({"Test, fine-tuning": test_result,
              "Test, lezione scorsa (SVM su vettori congelati)": {"Macro-F1": 0.672, "negativo": 0.610,
                                                                 "neutro": 0.471, "positivo": 0.935}}).T.round(3)

Modello scelto: Specializzato, fine-tuning. Migliore in assoluto in validazione: Specializzato, fine-tuning


,Macro-F1,negativo,neutro,positivo
"Test, fine-tuning",0.712,0.659,0.528,0.949
"Test, lezione scorsa (SVM su vettori congelati)",0.672,0.610,0.471,0.935


In [26]:
pd.crosstab(test_df["sentiment"], pd.Categorical(np.array(CLASSES)[test_pred], categories=CLASSES),
            rownames=["Reale"], colnames=["Previsto"])

Previsto,negativo,neutro,positivo
Reale,,,
negativo,296,150,28
neutro,105,312,148
positivo,23,154,3311


- Sul test il fine-tuning supera il modello della lezione scorsa in **tutte e tre** le classi,
  soprattutto negativo e neutro, pur avendo visto meno recensioni.
- Il neutro resta la classe più difficile: gli errori finiscono **in entrambe** le classi vicine,
  perché una recensione a 3 stelle può pendere da una parte o dall'altra.

Salviamo tutti i risultati in un unico file: è la memoria dell'esperimento (e la fonte dei numeri delle slide).

In [27]:
summary = {"validazione": results, "curva": curve_df.to_dict("records"), "costi": costs.to_dict(),
           "test": test_result, "modello_test": best_name, "griglia": grid.to_dict(),
           "storie": histories, "best_C_tfidf": best_C,
           "frasi": {name: examples_table[name].to_dict("index") for name in rows},
           "file_modello_MB": model_files.to_dict(), "file_checkpoint_MB": checkpoint_files.to_dict(),
           "gpu": torch.cuda.get_device_name(0) if DEVICE == "cuda" else "CPU"}
(SAVE_DIR / "risultati_esercizio_1.json").write_text(json.dumps(summary, indent=1, default=float))
print("Salvato:", SAVE_DIR / "risultati_esercizio_1.json")

Salvato: salvataggi/risultati_esercizio_1.json


## 9. Cosa portare a casa

1. **Congelare** è facile e veloce: i vettori di RoBERTa battono TF-IDF di poco, quelli di un modello di embedding nettamente.
2. **Il fine-tuning** adatta il modello al nostro compito e migliora soprattutto le classi difficili;
   partendo dal modello specializzato dà il risultato migliore.
3. **Con pochi dati** il punto di partenza conta di più: un modello già specializzato impara più in fretta,
   e un **modello di embedding moderno**, anche congelato, vale quanto un fine-tuning a una frazione del costo.
4. I modelli si scelgono con le **classifiche** (come MTEB) e si confermano sui **propri dati**.
5. **Salvare** modello, tokenizer e (durante l'addestramento) i checkpoint fa parte del lavoro.
6. **Prestazioni e costi** vanno misurati insieme.

**Per il project work**: cercate sull'Hugging Face Hub un modello già addestrato su un compito simile al vostro,
misuratelo pronto all'uso, poi provatelo congelato e con il fine-tuning, partendo da poche centinaia di esempi etichettati.